In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QUICK-PDE: функція Qiskit від ColibriTD
> **Note:** Функції Qiskit — це експериментальна можливість, доступна користувачам IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan. Вони перебувають у статусі попереднього релізу та можуть змінюватись.
## Огляд
Розв'язувач диференціальних рівнянь у частинних похідних (PDE), представлений тут, є частиною нашої платформи Quantum Innovative Computing Kit (QUICK) (QUICK-PDE) і запакований як функція Qiskit. За допомогою функції QUICK-PDE ти можеш розв'язувати предметно-орієнтовані диференціальні рівняння в частинних похідних на QPU IBM Quantum. Ця функція базується на алгоритмі, описаному в [статті ColibriTD про H-DES.](https://arxiv.org/abs/2410.01130) Цей алгоритм може розв'язувати складні мультифізичні задачі, починаючи з обчислювальної гідродинаміки (CFD) та деформації матеріалів (MD), і незабаром з'являться нові варіанти використання.

Щоб впоратися з диференціальними рівняннями, пробні розв'язки кодуються як лінійні комбінації ортогональних функцій (зазвичай поліноми Чебишева, і конкретніше $2^n$ таких, де $n$ — кількість кубітів, що кодують твою функцію), параметризовані кутами Змінного Квантового Ланцюга (VQC). Анзац генерує стан, що кодує функцію, яку обчислюють спостережувані величини, комбінації яких дозволяють обчислити функцію у всіх точках. Потім ти можеш обчислити функцію втрат, у якій закодовано диференціальні рівняння, та налаштувати кути в гібридному циклі, як показано нижче. Пробні розв'язки поступово наближаються до фактичних розв'язків, доки не буде досягнуто задовільного результату.

![Робочий процес функції QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

На додаток до цього гібридного циклу, ти також можеш поєднувати різні оптимізатори в ланцюжок. Це корисно, коли потрібно, щоб глобальний оптимізатор знайшов хороший набір кутів, а потім точніший оптимізатор пішов за градієнтом до найкращого набору сусідніх кутів. У випадку обчислювальної гідродинаміки (CFD) стандартна послідовність оптимізації дає найкращі результати — але у випадку деформації матеріалів (MD), хоча стандартні налаштування і дають хороші результати, ти можеш налаштувати її додатково для переваг, специфічних для конкретної задачі.

Зауваж, що для кожної змінної функції ми вказуємо кількість кубітів (з якою ти можеш експериментувати). Стекуючи 10 ідентичних схем та обчислюючи 10 ідентичних спостережуваних на різних кубітах в одній великій схемі, ти можеш знижувати шуми в процесі CMA-оптимізації, спираючись на метод навчання шумів, та значно зменшити потрібну кількість знімків (shots).

## Вхідні/вихідні дані
### Обчислювальна гідродинаміка
Невʼязке рівняння Бюргерса моделює потік невʼязкої рідини таким чином:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ представляє поле швидкості рідини. Цей варіант використання має часову граничну умову: ти можеш задати початкову умову, а потім дозволити системі розслабитись. Наразі єдиними прийнятними початковими умовами є лінійні функції: $ax + b$.

Аргументи диференціальних рівнянь CFD знаходяться на фіксованій сітці, таким чином:

- $t$ знаходиться між 0 і 0.95 з 30 точками вибірки. $x$ знаходиться між 0 і 0.95 з кроком 0.2375.

### Деформація матеріалів
Цей варіант використання зосереджений на гіпопружній деформації з одновимірним випробуванням на розтяг, у якому стержень, закріплений в просторі, тягнуть за інший кінець. Ми описуємо задачу таким чином:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ представляє об'ємний модуль матеріалу, що розтягується, $n$ — показник степеневого закону, $b$ — силу на одиницю маси, $\epsilon_0$ — межу пропорційного напруження, $\sigma_0$ — межу пропорційної деформації, $u$ — функцію напруження, а $\sigma$ — функцію деформації.

Розглянутий стержень має одиничну довжину. Цей варіант використання має граничну умову для поверхневого напруження $t$, тобто кількість роботи, необхідної для розтягування стержня.

Аргументи диференціальних рівнянь MD знаходяться на фіксованій сітці, таким чином:

- $x$ знаходиться між 0 і 1 з кроком 0.04.

### Вхідні дані
Щоб запустити функцію Qiskit QUICK-PDE, ти можеш налаштувати такі параметри:

| Назва             | Тип                                                  | Опис                                                                                                                                                                                                                                                                            | Специфічний для варіанту | Приклад                  |
| ----------------- | ---------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------ | ------------------------ |
| use_case          | `Literal["MD", "CFD"]`                               | Перемикач для вибору системи диференціальних рівнянь для розв'язання                                                                                                                                                                                                            | Ні                       | `"CFD"`                  |
| parameters        | `dict[str, Any]`                                     | Параметри диференціального рівняння (докладніше дивись у наступній таблиці)                                                                                                                                                                                                     | Ні                       | `{"a": 1.0, "b": 1.0}`   |
| nb_qubits         | `Optional[dict[str, dict[str, int]]]`                | Кількість кубітів на функцію та на змінну. Функція вибирає оптимізовані значення, але якщо ти хочеш спробувати знайти кращу комбінацію, можеш перевизначити стандартні значення                                                                                                 | Ні                       | `{"u": {"x": 1, "t":3}}` |
| depth             | `Optional[dict[str, int]]`                           | Глибина анзацу на функцію. Функція вибирає оптимізовані значення, але якщо ти хочеш спробувати знайти кращу комбінацію, можеш перевизначити стандартні значення                                                                                                                 | Ні                       | `{"u": 4}`               |
| optimizer         | `Optional[list[str]]`                                | Оптимізатори для використання: або "CMAES" з [бібліотеки Python `cma`](https://github.com/CMA-ES/pycma), або один з [оптимізаторів scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)                                                    | MD                       | `"SLSQP"`                |
| shots             | `Optional[list[int]]`                                | Кількість знімків для запуску кожної схеми. Оскільки потрібно кілька кроків оптимізації, довжина списку має дорівнювати кількості використовуваних оптимізаторів (4 для CFD). За замовчуванням `[50_000] * nb_optimizers` для MD і `[5_00, 2_000, 5_000, 10_000]` для CFD        | Ні                       | `[15_000, 30_000]`       |
| optimizer_options | `Optional[dict[str, Any]]`                           | Параметри для передачі оптимізатору. Деталі цього вхідного значення залежать від використовуваного оптимізатора; для подробиць звертайся до документації відповідного оптимізатора                                                                                               | MD                       | `{"maxiter": 50 }`       |
| initialization    | `Optional[Literal["RANDOM", "PHYSICALLY_INFORMED"]]` | Чи починати з випадкових кутів, чи з розумно обраних. Зауваж, що розумно обрані кути можуть не спрацювати в 100% випадків. За замовчуванням `"PHYSICALLY_INFORMED"`.                                                                                                            | Ні                       | `"RANDOM"`               |
| backend_name      | `Optional[str]`                                      | Ім'я бекенду для використання.                                                                                                                                                                                                                                                  | Ні                       | `"ibm_torino"`           |
| mode              | `Optional[Literal["job", "session", "batch"]]`        | Режим виконання для використання. За замовчуванням `"job"`.                                                                                                                                                                                                                     | Ні                       | `"job"`                  |

Параметри диференціального рівняння (фізичні параметри та гранична умова) мають відповідати такому формату:

| Варіант | Ключ        | Тип значення | Опис                                     | Приклад |
| ------- | ----------- | ------------ | ---------------------------------------- | ------- |
| CFD     | `a`         | `float`      | Коефіцієнт початкових значень $u$        | `1.0`   |
| CFD     | `b`         | `float`      | Зміщення початкових значень $u$          | `1.0`   |
| MD      | `t`         | `float`      | поверхневе напруження                    | `12.0`  |
| MD      | `K`         | `float`      | об'ємний модуль                          | `100.0` |
| MD      | `n`         | `int`        | степеневий закон                         | `4.0`   |
| MD      | `b`         | `float`      | сила на одиницю маси                     | `10.0`  |
| MD      | `epsilon_0` | `float`      | межа пропорційного напруження            | `0.1`   |
| MD      | `sigma_0`   | `float`      | межа пропорційної деформації             | `5.0`   |

### Вихідні дані
Вихідні дані — це словник зі списком точок вибірки, а також значеннями функцій у кожній з цих точок:

In [ ]:
from numpy import array

In [ ]:
solution = {
    "functions": {
        "u": array(
            [
                [0.01, 0.1, 1],
                [0.02, 0.2, 2],
                [0.03, 0.3, 3],
                [0.04, 0.4, 4],
            ]
        ),
    },
    "samples": {
        "t": array([0.1, 0.2, 0.3, 0.4]),
        "x": array([0.5, 0.6, 0.7]),
    },
}

Форма масиву розв'язку залежить від вибірок змінних:

In [ ]:
assert len(solution["functions"]["u"].shape) == len(solution["samples"])
for col_size, samples in zip(
    solution["functions"]["u"].shape, solution["samples"].values()
):
    assert col_size == len(samples)

Відображення між точками вибірки змінних функції та виміром масиву розв'язку виконується в алфавітно-числовому порядку імені змінної. Наприклад, якщо змінними є `"t"` і `"x"`, рядок `solution["functions"]["u"]` представляє значення розв'язку для фіксованого `"t"`, а стовпець `solution["functions"]["u"]` представляє значення розв'язку для фіксованого `"x"`.

Нижче наведено приклад того, як отримати значення функції для конкретного набору координат:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Бенчмарки

У наступній таблиці представлено статистику різних запусків нашої функції.

| Приклад                            | Кількість кубітів | Ініціалізація         | Похибка   | Загальний час (хв) | Використання середовища виконання (хв) |
| ---------------------------------- | ----------------- | --------------------- | --------- | ------------------ | --------------------------------------- |
| Невʼязке рівняння Бюргерса         | 50                | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                 | 25                                      |
| Гіпопружне одновимірне розтягнення | 18                | `RANDOM`              | $10^{-2}$ | 123                | 100                                     |

## Початок роботи

Заповни [форму для запиту доступу до функції QUICK-PDE.](https://forms.cloud.microsoft/e/3Wi9cbjQPK) Потім, якщо ти вже [зберіг свій обліковий запис](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі, вибери функцію таким чином:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## Приклади

Щоб розпочати, спробуй один із наступних прикладів:

### Обчислювальна гідродинаміка (CFD)

Коли початкові умови задані як $u(0,x) = x$, результати такі:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Перевір [статус](/guides/functions#check-job-status) або отримай [результати](/guides/functions#retrieve-results) навантаження своєї функції Qiskit таким чином:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material Deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![Вихідні дані попередньої комірки коду](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Деформація матеріалів
Варіант використання деформації матеріалів вимагає фізичних параметрів твого матеріалу та прикладеної сили, таким чином:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



![Вихідні дані попередньої комірки коду](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

## Отримання повідомлень про помилки

Якщо статус твого навантаження — `ERROR`, використай `job.error_message()`, щоб отримати повідомлення про помилку для налагодження, таким чином: